In [1]:
   import pandas as pd
   print("Setup works, pandas version:", pd.__version__)

Setup works, pandas version: 3.0.6


In [2]:
from pathlib import Path

print("Working folder:", Path.cwd())
print()
for p in Path.cwd().parent.joinpath("data", "raw").glob("*"):
    print(repr(p.name))

Working folder: c:\Users\Mohib\price-intelligence-pipeline\notebooks\notebooks



In [4]:
from pathlib import Path
import pandas as pd

raw_dir = Path.cwd().parents[1] / "data" / "raw"
print("Looking in:", raw_dir)
print([p.name for p in raw_dir.glob("*")])

csv_path = next(raw_dir.glob("*.csv"))
df = pd.read_csv(csv_path)
print("Loaded:", csv_path.name)
print(df.shape)
df.head()

Looking in: c:\Users\Mohib\price-intelligence-pipeline\data\raw
['BigBasket Products.csv']
Loaded: BigBasket Products.csv
(27555, 10)


,index,product,category,sub_category,brand,sale_price,market_price,type,rating,description
0,1,Garlic Oil - Vegetarian Capsule 500 mg,Beauty & Hygiene,Hair Care,Sri Sri Ayurveda,220.0,220.0,Hair Oil & Serum,4.1,This Product contains Garlic Oil that is known...
1,2,Water Bottle - Orange,"Kitchen, Garden & Pets",Storage & Accessories,Mastercook,180.0,180.0,Water & Fridge Bottles,2.3,"Each product is microwave safe (without lid), ..."
2,3,"Brass Angle Deep - Plain, No.2",Cleaning & Household,Pooja Needs,Trm,119.0,250.0,Lamp & Lamp Oil,3.4,"A perfect gift for all occasions, be it your m..."
3,4,Cereal Flip Lid Container/Storage Jar - Assort...,Cleaning & Household,Bins & Bathroom Ware,Nakoda,149.0,176.0,"Laundry, Storage Baskets",3.7,Multipurpose container with an attractive desi...
4,5,Creme Soft Soap - For Hands & Body,Beauty & Hygiene,Bath & Hand Wash,Nivea,162.0,162.0,Bathing Bars & Soaps,4.4,Nivea Creme Soft Soap gives your skin the best...


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 27555 entries, 0 to 27554
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   index         27555 non-null  int64  
 1   product       27554 non-null  str    
 2   category      27555 non-null  str    
 3   sub_category  27555 non-null  str    
 4   brand         27554 non-null  str    
 5   sale_price    27555 non-null  float64
 6   market_price  27555 non-null  float64
 7   type          27555 non-null  str    
 8   rating        18929 non-null  float64
 9   description   27440 non-null  str    
dtypes: float64(3), int64(1), str(6)
memory usage: 2.1 MB


In [6]:
df.isna().sum()

index              0
product            1
category           0
sub_category       0
brand              1
sale_price         0
market_price       0
type               0
rating          8626
description      115
dtype: int64

In [7]:
df.describe()

,index,sale_price,market_price,rating
count,27555.00000,27555.000000,27555.000000,18929.000000
mean,13778.00000,322.514808,382.056664,3.943410
std,7954.58767,486.263116,581.730717,0.739063
min,1.00000,2.450000,3.000000,1.000000
25%,6889.50000,95.000000,100.000000,3.700000
50%,13778.00000,190.000000,220.000000,4.100000
75%,20666.50000,359.000000,425.000000,4.300000
max,27555.00000,12500.000000,12500.000000,5.000000


In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
print("sale > market:", (df.sale_price > df.market_price).sum())
print("sale == market (no discount):", (df.sale_price == df.market_price).sum())
print()
print(df.category.value_counts())
print()
print(df.nlargest(5, "sale_price")[["product", "brand", "sale_price", "market_price"]])

sale > market: 0
sale == market (no discount): 12326

category
Beauty & Hygiene            7867
Gourmet & World Food        4690
Kitchen, Garden & Pets      3580
Snacks & Branded Foods      2814
Foodgrains, Oil & Masala    2676
Cleaning & Household        2675
Beverages                    885
Bakery, Cakes & Dairy        851
Baby Care                    610
Fruits & Vegetables          557
Eggs, Meat & Fish            350
Name: count, dtype: int64

                                                 product     brand  \
25301                                    Bravura Clipper      Wahl   
21761      Pet Food - N&D Team Breeder Puppy Top Farmina   Farmina   
12669                   Epilator SE9-9961 Legs-Body-Face     Braun   
23082  Gas Stove-4 Burner Royale Plus Schott Glass, B...  Prestige   
2781                              Extra Virgin Olive Oil    Abbies   

       sale_price  market_price  
25301    12500.00       12500.0  
21761    10090.00       10090.0  
12669     8184.44       

In [10]:
from pathlib import Path
from datetime import date

# find the project root (works wherever the notebook lives)
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())

clean = df.drop(columns="index").dropna(subset=["product", "brand"]).reset_index(drop=True)

for col in ["product", "category", "sub_category", "brand", "type"]:
    clean[col] = clean[col].str.strip()

# drop rows where the sale price is above the market price (invalid)
before = len(clean)
clean = clean[clean.sale_price <= clean.market_price].reset_index(drop=True)
print("Dropped for bad prices:", before - len(clean))

clean["product_id"] = clean.index + 1
clean["discount_pct"] = ((clean.market_price - clean.sale_price) / clean.market_price * 100).round(1)

products = clean[["product_id", "product", "category", "sub_category", "brand", "type", "rating", "description"]]

snapshots = clean[["product_id", "sale_price", "market_price", "discount_pct"]].copy()
snapshots["snapshot_date"] = date.today().isoformat()
snapshots["source"] = "bigbasket_kaggle_baseline"

print(products.shape, snapshots.shape)

Dropped for bad prices: 0
(27553, 8) (27553, 6)


In [11]:
import sqlite3

db_path = root / "data" / "processed" / "prices.db"
conn = sqlite3.connect(db_path)
products.to_sql("products", conn, if_exists="replace", index=False)
snapshots.to_sql("price_snapshots", conn, if_exists="replace", index=False)
conn.close()
print("Saved:", db_path)

Saved: c:\Users\Mohib\price-intelligence-pipeline\data\processed\prices.db


In [12]:
conn = sqlite3.connect(db_path)
q = """
SELECT p.category,
       COUNT(*) AS products,
       ROUND(AVG(s.discount_pct), 1) AS avg_discount_pct
FROM products p
JOIN price_snapshots s USING (product_id)
GROUP BY p.category
ORDER BY avg_discount_pct DESC;
"""
pd.read_sql(q, conn)

,category,products,avg_discount_pct
0,"Kitchen, Garden & Pets",3580,22.2
1,Fruits & Vegetables,557,21.2
2,Beauty & Hygiene,7867,12.3
3,"Foodgrains, Oil & Masala",2676,11.8
4,Cleaning & Household,2674,10.8
5,Beverages,884,9.6
6,"Eggs, Meat & Fish",350,8.1
7,Gourmet & World Food,4690,7.9
8,"Bakery, Cakes & Dairy",851,7.7
9,Snacks & Branded Foods,2814,6.6


In [13]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(db_path)

def q(sql):
    return pd.read_sql(sql, conn)

In [14]:
q("""
SELECT COUNT(*) AS total_products,
       SUM(discount_pct > 0) AS discounted_products,
       ROUND(100.0 * SUM(discount_pct > 0) / COUNT(*), 1) AS pct_discounted,
       ROUND(AVG(CASE WHEN discount_pct > 0 THEN discount_pct END), 1) AS avg_discount_when_discounted
FROM price_snapshots;
""")

,total_products,discounted_products,pct_discounted,avg_discount_when_discounted
0,27553,15228,55.3,21.4


In [15]:
q("""
SELECT p.category,
       COUNT(*) AS products,
       ROUND(100.0 * SUM(s.discount_pct > 0) / COUNT(*), 1) AS pct_discounted,
       ROUND(AVG(CASE WHEN s.discount_pct > 0 THEN s.discount_pct END), 1) AS avg_discount_when_discounted
FROM products p
JOIN price_snapshots s USING (product_id)
GROUP BY p.category
ORDER BY avg_discount_when_discounted DESC;
""")

,category,products,pct_discounted,avg_discount_when_discounted
0,"Kitchen, Garden & Pets",3580,77.5,28.6
1,Fruits & Vegetables,557,97.5,21.8
2,Beverages,884,44.3,21.6
3,Beauty & Hygiene,7867,59.2,20.9
4,"Foodgrains, Oil & Masala",2676,59.9,19.8
5,Gourmet & World Food,4690,40.4,19.5
6,Cleaning & Household,2674,55.5,19.4
7,Snacks & Branded Foods,2814,37.8,17.6
8,Baby Care,610,36.2,16.2
9,"Eggs, Meat & Fish",350,50.3,16.0


In [16]:
q("""
SELECT p.brand,
       COUNT(*) AS products,
       ROUND(AVG(s.discount_pct), 1) AS avg_discount_pct
FROM products p
JOIN price_snapshots s USING (product_id)
GROUP BY p.brand
HAVING COUNT(*) >= 20
ORDER BY avg_discount_pct DESC
LIMIT 10;
""")

,brand,products,avg_discount_pct
0,Nova,23,57.9
1,Frestol,28,52.1
2,Vinayak,20,48.5
3,Trm,42,46.7
4,Steelo,28,44.3
5,Laplast,37,42.2
6,Mirabelle,25,40.3
7,BORGES,50,39.5
8,Nirlon,22,39.0
9,Flair,48,38.7


In [17]:
q("""
SELECT CASE
         WHEN discount_pct = 0  THEN '0% (no discount)'
         WHEN discount_pct < 10 THEN '1-9%'
         WHEN discount_pct < 25 THEN '10-24%'
         WHEN discount_pct < 50 THEN '25-49%'
         ELSE '50%+'
       END AS band,
       COUNT(*) AS products
FROM price_snapshots
GROUP BY band
ORDER BY MIN(discount_pct);
""")

,band,products
0,0% (no discount),12325
1,1-9%,2606
2,10-24%,7071
3,25-49%,4808
4,50%+,743


In [18]:
import json

QUERIES = {
    "overall": """
        SELECT COUNT(*) AS total_products,
               SUM(discount_pct > 0) AS discounted_products,
               ROUND(100.0 * SUM(discount_pct > 0) / COUNT(*), 1) AS pct_discounted,
               ROUND(AVG(CASE WHEN discount_pct > 0 THEN discount_pct END), 1) AS avg_discount_when_discounted
        FROM price_snapshots;""",
    "categories": """
        SELECT p.category, COUNT(*) AS products,
               ROUND(100.0 * SUM(s.discount_pct > 0) / COUNT(*), 1) AS pct_discounted,
               ROUND(AVG(CASE WHEN s.discount_pct > 0 THEN s.discount_pct END), 1) AS avg_discount_when_discounted
        FROM products p JOIN price_snapshots s USING (product_id)
        GROUP BY p.category ORDER BY avg_discount_when_discounted DESC;""",
    "top_brands": """
        SELECT p.brand, COUNT(*) AS products, ROUND(AVG(s.discount_pct), 1) AS avg_discount_pct
        FROM products p JOIN price_snapshots s USING (product_id)
        GROUP BY p.brand HAVING COUNT(*) >= 20
        ORDER BY avg_discount_pct DESC LIMIT 10;""",
}

facts = {name: json.loads(q(sql).to_json(orient="records")) for name, sql in QUERIES.items()}
print(json.dumps(facts, indent=2)[:800])

{
  "overall": [
    {
      "total_products": 27553,
      "discounted_products": 15228,
      "pct_discounted": 55.3,
      "avg_discount_when_discounted": 21.4
    }
  ],
  "categories": [
    {
      "category": "Kitchen, Garden & Pets",
      "products": 3580,
      "pct_discounted": 77.5,
      "avg_discount_when_discounted": 28.6
    },
    {
      "category": "Fruits & Vegetables",
      "products": 557,
      "pct_discounted": 97.5,
      "avg_discount_when_discounted": 21.8
    },
    {
      "category": "Beverages",
      "products": 884,
      "pct_discounted": 44.3,
      "avg_discount_when_discounted": 21.6
    },
    {
      "category": "Beauty & Hygiene",
      "products": 7867,
      "pct_discounted": 59.2,
      "avg_discount_when_discounted": 20.9
    },
    {
      "cat


In [19]:
import os, requests
from dotenv import load_dotenv

load_dotenv(root / ".env")
API_KEY = os.getenv("GROQ_API_KEY")
MODEL = "GPT OSS 120B"

system_msg = (
    "You are a retail pricing analyst. Use ONLY the numbers in the data provided. "
    "Do not invent figures, causes or trends over time. The data is a single price snapshot."
)
user_msg = (
    "Write a 5-sentence summary for a business manager, then 2 recommendations. "
    f"Data (JSON): {json.dumps(facts)}"
)

resp = requests.post(
    "https://api.groq.com/openai/v1/chat/completions",
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ],
        "temperature": 0.2,
    },
    timeout=60,
)
resp.raise_for_status()
print(resp.json()["choices"][0]["message"]["content"])

HTTPError: 404 Client Error: Not Found for url: https://api.groq.com/openai/v1/chat/completions

In [20]:
r = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {API_KEY}"},
    timeout=30,
)
print(r.status_code)
for m in r.json().get("data", []):
    print(m["id"])

200
whisper-large-v3-turbo
qwen/qwen3.8-27b
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-86m
canopylabs/orpheus-v1-english
whisper-large-v3
canopylabs/orpheus-arabic-saudi
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-20b
allam-2-7b
openai/gpt-oss-120b


In [21]:
MODEL = "openai/gpt-oss-120b"

In [22]:
resp = requests.post(
    "https://api.groq.com/openai/v1/chat/completions",
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ],
        "temperature": 0.2,
    },
    timeout=60,
)

if resp.status_code != 200:
    print(resp.status_code, resp.text)
else:
    print(resp.json()["choices"][0]["message"]["content"])

**Summary for the Business Manager**

1. Across the entire assortment of 27,553 SKUs, 15,228 items (55.3 %) are currently discounted, with an average discount of 21.4 % on those items.  
2. The “Fruits & Vegetables” category has the highest discount penetration at 97.5 % of its 557 SKUs, while “Kitchen, Garden & Pets” applies the deepest average discount at 28.6 % across 3,580 SKUs.  
3. Categories such as “Snacks & Branded Foods” (37.8 % discounted) and “Baby Care” (36.2 % discounted) show relatively lower discount activity, with average discounts of 17.6 % and 16.2 % respectively.  
4. Among brands, Nova leads with the steepest average discount of 57.9 % on 23 SKUs, followed by Frestol (52.1 % on 28 SKUs) and Vinayak (48.5 % on 20 SKUs).  
5. Overall, discount intensity varies widely by category and brand, indicating opportunities to fine‑tune pricing strategies where discount levels are either unusually high or low.

**Recommendations**

1. **Target high‑penetration, low‑discount ca